# PCG-to-ECG Signal Reconstruction: Tanh Gradient Analysis

**Ablation Study Code** for: *PCG-to-ECG Signal Reconstruction Using Conditional GAN with Multi-Scale Attention* (EUSIPCO 2026)

**Authors:** Zafar Ahmad, Nazia Perwaiz, Muhammad Salman Khan

---

This notebook reproduces the Tanh gradient vanishing analysis from the ablation study (Trail-03 golden discussion).

**Core finding:**
```
tanh'(0.5) = 0.786   (baseline region, gradient flows)
tanh'(3.0) = 0.010   (R-peak region,  gradient vanishes)
Ratio      = 80x
```

R-peaks in z-scored ECG reach |z| = 3–4. At these amplitudes the Tanh derivative is near zero, so the model receives almost no learning signal for R-peak reconstruction.

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

---
## Gradient Functions

In [ ]:
def tanh_gradient(x):
    """
    Compute d/dx tanh(x) = sech^2(x) = 1 - tanh^2(x).
    """
    return 1.0 - np.tanh(x) ** 2


def batchnorm_gradient(gamma=1.0, var=1.0, eps=1e-5):
    """
    Approximate gradient of BatchNorm w.r.t. its input.
    d BN(z) / dz  ~  gamma / sqrt(var + eps)

    Unlike Tanh, this value is constant across all amplitudes.
    """
    return gamma / np.sqrt(var + eps)

---
## Gradient Ratio Table

ECG amplitude regions after z-score normalization:
- |z| < 0.5: isoelectric baseline (~50% of signal)
- |z| = 0.5–1: P-wave, T-wave (~20%)
- |z| = 1–2: QRS onset / offset (~20%)
- |z| > 2: R-peak (~10%)

In [ ]:
amplitudes = [0.0, 0.25, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0]
regions = {
    0.0: "isoelectric",
    0.25: "isoelectric",
    0.5: "baseline",
    1.0: "P/T wave",
    1.5: "QRS onset",
    2.0: "QRS complex",
    2.5: "R-peak base",
    3.0: "R-peak",
    3.5: "R-peak tip",
    4.0: "R-peak tip",
}

bn_grad = batchnorm_gradient()

print(f"{'|z|':>5s}  {'Region':<15s}  {'tanh grad':>10s}  "
      f"{'BN grad':>8s}  {'Ratio BN/tanh':>14s}")
print("-" * 60)

for z in amplitudes:
    tg = tanh_gradient(z)
    ratio = bn_grad / tg if tg > 1e-6 else float('inf')
    print(f"{z:5.2f}  {regions[z]:<15s}  {tg:10.6f}  "
          f"{bn_grad:8.4f}  {ratio:14.1f}x")

---
## Visualization: Tanh vs BatchNorm Gradient

In [ ]:
x = np.linspace(0, 5, 500)
tanh_g = tanh_gradient(x)
bn_g = np.ones_like(x) * batchnorm_gradient()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Left: Tanh function and its derivative
ax1.plot(x, np.tanh(x), 'b-', linewidth=2, label='tanh(x)')
ax1.plot(x, tanh_g, 'r--', linewidth=2, label="tanh'(x)")
ax1.axvline(x=3.0, color='gray', linestyle=':', alpha=0.7)
ax1.annotate('R-peak region\n|z| = 3.0', xy=(3.0, 0.01),
             xytext=(3.5, 0.4), fontsize=10,
             arrowprops=dict(arrowstyle='->', color='red'))
ax1.set_xlabel('|z| (amplitude)', fontsize=12)
ax1.set_ylabel('Value', fontsize=12)
ax1.set_title('Tanh: Output saturates, gradient vanishes', fontsize=11)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_ylim(-0.05, 1.1)

# Right: Gradient comparison
ax2.semilogy(x, tanh_g, 'r-', linewidth=2, label="Tanh gradient")
ax2.semilogy(x, bn_g, 'g-', linewidth=2, label="BatchNorm gradient")
ax2.axvspan(2.5, 4.5, alpha=0.15, color='red', label='R-peak zone')
ax2.axvspan(0, 1.0, alpha=0.15, color='blue', label='Baseline zone')
ax2.set_xlabel('|z| (amplitude)', fontsize=12)
ax2.set_ylabel('Gradient magnitude (log scale)', fontsize=12)
ax2.set_title('80x gradient gap at R-peaks', fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_ylim(1e-4, 2)

plt.tight_layout()
plt.savefig('tanh_gradient_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: tanh_gradient_analysis.png")

---
## PSNR Bias Decomposition

ECG signal composition (z-score normalized):
- ~70% of samples have |z| < 1 (baseline + P/T waves)
- ~20% have 1 < |z| < 2 (QRS onset/offset)
- ~10% have |z| > 2 (R-peaks)

A model with Tanh output clips R-peaks to +/- 1. The clipping error is large, but only on 10% of samples. PSNR averages MSE across ALL samples, so 70% accurate baseline dominates and masks the R-peak failure.

In [ ]:
print("=== PSNR Bias Decomposition ===")
print()

# Per-region errors (Tanh-clipped model)
regions_data = [
    ("Baseline (70%)", 0.70, 0.20),   # low error
    ("QRS (20%)",      0.20, 0.50),   # medium error
    ("R-peaks (10%)",  0.10, 3.00),   # catastrophic error
]

total_mse = 0.0
print(f"{'Region':<20s}  {'Weight':>6s}  {'|error|':>8s}  "
      f"{'MSE contrib':>12s}")
print("-" * 52)
for name, weight, err in regions_data:
    contrib = weight * err ** 2
    total_mse += contrib
    print(f"{name:<20s}  {weight:6.0%}  {err:8.2f}  {contrib:12.4f}")

max_val = 4.0  # typical max |z| in ECG
psnr = 20 * np.log10(max_val / np.sqrt(total_mse))

print(f"\n  Total MSE     = {total_mse:.4f}")
print(f"  max|signal|   = {max_val}")
print(f"  PSNR          = {psnr:.1f} dB")

r_peak_pct = regions_data[2][1] * regions_data[2][2]**2 / total_mse
baseline_pct = regions_data[0][1] * regions_data[0][2]**2 / total_mse
print(f"\n  R-peak region contributes {r_peak_pct:.0%}"
      f" of MSE but occupies only 10% of samples.")
print(f"  Baseline contributes {baseline_pct:.0%}"
      f" of MSE while occupying 70% of samples.")

---
## Visualization: MSE Contribution by Region

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

# Left: Signal composition (% of samples)
labels = ['Baseline\n(|z| < 1)', 'QRS\n(1 < |z| < 2)', 'R-peaks\n(|z| > 2)']
sample_pcts = [70, 20, 10]
colors = ['#4ECDC4', '#FFD93D', '#FF6B6B']
ax1.bar(labels, sample_pcts, color=colors, edgecolor='black', linewidth=0.8)
ax1.set_ylabel('% of signal samples', fontsize=12)
ax1.set_title('Signal Composition', fontsize=12)
ax1.set_ylim(0, 85)
for i, v in enumerate(sample_pcts):
    ax1.text(i, v + 2, f'{v}%', ha='center', fontsize=11, fontweight='bold')

# Right: MSE contribution
mse_contribs = [0.028, 0.050, 0.900]
mse_pcts = [c / sum(mse_contribs) * 100 for c in mse_contribs]
ax2.bar(labels, mse_pcts, color=colors, edgecolor='black', linewidth=0.8)
ax2.set_ylabel('% of total MSE', fontsize=12)
ax2.set_title('MSE Contribution (Tanh-clipped model)', fontsize=12)
ax2.set_ylim(0, 105)
for i, v in enumerate(mse_pcts):
    ax2.text(i, v + 2, f'{v:.0f}%', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('psnr_bias_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: psnr_bias_decomposition.png")
print("\nR-peaks are 10% of samples but cause 92% of MSE.")
print("PSNR averages over all samples, hiding this concentration.")